# SalesMesh Analytics Dashboard — ML Pipeline

**A Marketing Lead Conversion Prediction Model and Pipeline Analytics Dashboard for Data-Driven Decision Making**

Okeyo Hoppe Solini Marie (161031) · Supervisor: Mr. Tiberius Tabulu · Strathmore University

This notebook:
1. Cleans the Kaggle **Lead Scoring** dataset
2. Compares **Logistic Regression**, **Random Forest** and **Gradient Boosting**, each tuned with **GridSearchCV** and **stratified 10-fold cross-validation**
3. Evaluates the selected model on a held-out 20% test set and analyses feature importance
4. Scores all 9,240 leads with a conversion probability and a **High / Medium / Low** priority tier
5. Cleans the Kaggle **Marketing Campaign Performance** dataset
6. Generates the SQL scripts that load both datasets into the `salesmesh` MySQL database for Power BI

**How to run:** Runtime → Change runtime type → **CPU** (scikit-learn does not use a GPU), then Runtime → Run all. You will be asked to upload `Lead Scoring.csv` (Section 2) and `marketing_campaign_dataset.csv` (Section 14). Model tuning in Section 7 takes several minutes.

## 1. Setup
Import the libraries used throughout the notebook.

In [ ]:
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
from google.colab import files

from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV, cross_val_predict
from sklearn.preprocessing import MinMaxScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (classification_report, confusion_matrix, roc_auc_score,
                             precision_score, recall_score, f1_score,
                             ConfusionMatrixDisplay, RocCurveDisplay)
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

RANDOM_STATE = 42

## 2. Load the Lead Scoring dataset
Upload `Lead Scoring.csv` when prompted. The expected shape is 9,240 rows × 37 columns.

In [ ]:
print("Upload: Lead Scoring.csv")
uploaded = files.upload()
df = pd.read_csv(list(uploaded.keys())[0])

print("Shape:", df.shape)

## 3. Explore the data
Check the target variable balance and which columns have missing values.

In [ ]:
print("Target variable distribution (Converted):")
print(df['Converted'].value_counts())

print("\nColumns with the most missing values:")
print(df.isnull().sum().sort_values(ascending=False).head(15))

## 4. Clean the data
- **Drop** identifier columns, columns with over 35% missing values, and low-variance yes/no columns
- **Impute** missing categorical values with the mode and missing numerical values with the median
- **Treat `Select`** as missing — it is a placeholder meaning the lead did not choose an option

In [ ]:
cols_to_drop = [
    'Prospect ID', 'Lead Number',                        # identifiers, not predictors
    'Lead Quality', 'Tags',                              # too many missing values
    'Asymmetrique Activity Index', 'Asymmetrique Profile Index',
    'Asymmetrique Activity Score', 'Asymmetrique Profile Score',
    'What matters most to you in choosing a course',
    'Lead Profile',
    'Search', 'Magazine', 'Newspaper Article', 'X Education Forums',
    'Newspaper', 'Digital Advertisement', 'Through Recommendations',
    'Receive More Updates About Our Courses', 'Update me on Supply Chain Content',
    'Get updates on DM Content', 'I agree to pay the amount through cheque',
    'A free copy of Mastering The Interview'             # low-variance yes/no columns
]
df_clean = df.drop(columns=cols_to_drop)

# Categorical gaps -> mode
for col in ['What is your current occupation', 'Country',
            'How did you hear about X Education', 'Specialization', 'City']:
    df_clean[col] = df_clean[col].fillna(df_clean[col].mode()[0])

# Numerical gaps -> median
for col in ['TotalVisits', 'Page Views Per Visit']:
    df_clean[col] = df_clean[col].fillna(df_clean[col].median())

# 'Select' placeholder -> missing -> mode
df_clean = df_clean.replace('Select', pd.NA)
for col in df_clean.select_dtypes(include='object').columns:
    df_clean[col] = df_clean[col].fillna(df_clean[col].mode()[0])

print("Shape after cleaning:", df_clean.shape)
print("Missing values remaining:", df_clean.isnull().sum().sum())

## 5. Encode and split
- **One-hot encode** categorical columns so the models can use them (15 columns → 132 features)
- **Split** 80% training / 20% test, stratified so both sets keep the same conversion ratio. The test set is set aside and is not used again until the final evaluation in Section 9.
- `Prospect ID` is kept aside so every prediction can be traced back to its lead

In [ ]:
lead_ids = df['Prospect ID'].reset_index(drop=True)

X = df_clean.drop(columns=['Converted'])
y = df_clean['Converted']
X_encoded = pd.get_dummies(X, drop_first=True).astype(float)
print("Features after encoding:", X_encoded.shape[1])

X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)
print("Training set:", X_train.shape)
print("Test set:", X_test.shape)
print("Training class balance:", np.bincount(y_train))

## 6. Define the three candidate models
Each model is wrapped in one pipeline: **Min-Max scaling → SMOTE → classifier**.

Putting scaling and SMOTE *inside* the pipeline matters. During cross-validation, the pipeline is refitted on each training fold, so the scaler and SMOTE only ever learn from that fold. The validation fold and the test set never influence them, which prevents data leakage. SMOTE is only applied while fitting, so predictions are always made on real data.

| Model | Role | Hyperparameters tuned |
|---|---|---|
| Logistic Regression | Simple, interpretable baseline | Regularisation strength `C` |
| Random Forest | Ensemble of decision trees | Number of trees, tree depth, minimum leaf size |
| Gradient Boosting | Sequential ensemble that corrects earlier errors | Number of trees, learning rate, tree depth |

In [ ]:
def make_pipeline_for(model):
    return ImbPipeline([
        ('scaler', MinMaxScaler()),
        ('smote', SMOTE(random_state=RANDOM_STATE)),
        ('model', model),
    ])

candidates = {
    'Logistic Regression': (
        LogisticRegression(max_iter=2000, random_state=RANDOM_STATE),
        {'model__C': [0.01, 0.1, 1, 10]}
    ),
    'Random Forest': (
        RandomForestClassifier(random_state=RANDOM_STATE),
        {'model__n_estimators': [100, 200],
         'model__max_depth': [10, 15, None],
         'model__min_samples_leaf': [1, 2]}
    ),
    'Gradient Boosting': (
        GradientBoostingClassifier(random_state=RANDOM_STATE),
        {'model__n_estimators': [100, 200],
         'model__learning_rate': [0.05, 0.1],
         'model__max_depth': [3, 5]}
    ),
}

# Stratified 10-fold: every fold keeps the same converted / not-converted ratio
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=RANDOM_STATE)

## 7. Tune each model with GridSearchCV and stratified 10-fold cross-validation
GridSearchCV tries every hyperparameter combination listed in Section 6. Each combination is scored by its **mean F1-score across the 10 validation folds**. F1 is used because it balances precision and recall, which matters on imbalanced data. Only the 80% training set is used here.

This is the slowest step in the notebook and may take several minutes.

In [ ]:
search_results = {}
for name, (model, grid) in candidates.items():
    print(f"Tuning {name}...")
    search = GridSearchCV(make_pipeline_for(model), grid, scoring='f1', cv=cv, n_jobs=-1)
    search.fit(X_train, y_train)
    search_results[name] = search
    print(f"  Best mean CV F1: {search.best_score_:.4f}")
    print(f"  Best parameters: {search.best_params_}\n")

## 8. Compare the models and select the final one
The final model is chosen by **mean cross-validation F1**, not by test-set results. This keeps the test set as an independent final check. Test-set scores for all three models are shown for reporting.

In [ ]:
rows = []
for name, search in search_results.items():
    best = search.best_estimator_
    i = search.best_index_
    test_pred = best.predict(X_test)
    test_prob = best.predict_proba(X_test)[:, 1]
    rows.append({
        'Model': name,
        'CV F1 (mean)': search.cv_results_['mean_test_score'][i],
        'CV F1 (std)': search.cv_results_['std_test_score'][i],
        'Test precision': precision_score(y_test, test_pred),
        'Test recall': recall_score(y_test, test_pred),
        'Test F1': f1_score(y_test, test_pred),
        'Test AUC-ROC': roc_auc_score(y_test, test_prob),
    })

comparison = pd.DataFrame(rows).set_index('Model').round(4)
print(comparison.to_string())

best_name = comparison['CV F1 (mean)'].idxmax()
final_model = search_results[best_name].best_estimator_
print(f"\nSelected model: {best_name}")
print("Parameters:", search_results[best_name].best_params_)

## 9. Evaluate the selected model on the held-out test set
The 20% test set has not been used for training, tuning or model selection, so these results show how the model performs on unseen leads.

Targets from NFR-03: **F1-score ≥ 0.75** and **AUC-ROC ≥ 0.80**.

In [ ]:
y_pred = final_model.predict(X_test)
y_prob = final_model.predict_proba(X_test)[:, 1]
test_f1 = f1_score(y_test, y_pred)
test_auc = roc_auc_score(y_test, y_prob)

print(f"--- {best_name.upper()} (held-out test set) ---")
print(classification_report(y_test, y_pred, target_names=['Not converted', 'Converted']))
print(f"AUC-ROC: {test_auc:.4f}")
print(f"\nNFR-03 F1 >= 0.75:      {'MET' if test_f1 >= 0.75 else 'NOT MET'} ({test_f1:.4f})")
print(f"NFR-03 AUC-ROC >= 0.80: {'MET' if test_auc >= 0.80 else 'NOT MET'} ({test_auc:.4f})")

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred, display_labels=['Not converted', 'Converted'], cmap='Blues', ax=axes[0])
axes[0].set_title(f'Confusion matrix: {best_name}')

for name, search in search_results.items():
    RocCurveDisplay.from_estimator(search.best_estimator_, X_test, y_test, name=name, ax=axes[1])
axes[1].plot([0, 1], [0, 1], linestyle='--', color='grey')
axes[1].set_title('ROC curves on the test set')

plt.tight_layout()
plt.savefig('model_evaluation.png', dpi=150)
plt.show()

## 10. Feature importance
The tuned Random Forest's built-in importance scores show which lead attributes most influence conversion predictions. Because categories were one-hot encoded, each category (for example one lead source) appears as its own feature.

In [ ]:
rf_tuned = search_results['Random Forest'].best_estimator_.named_steps['model']
importances = pd.Series(rf_tuned.feature_importances_, index=X_train.columns).sort_values(ascending=False)

print("Top 15 features:")
print(importances.head(15).round(4).to_string())

fig, ax = plt.subplots(figsize=(9, 6))
importances.head(15).sort_values().plot.barh(ax=ax, color='#185FA5')
ax.set_title('Top 15 features: Random Forest importance')
ax.set_xlabel('Importance')
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150)
plt.show()

## 11. Score all leads and assign priority tiers
Every lead gets a probability of converting, P(Converted = 1). Scores are generated with **out-of-fold prediction**: the data is split into 10 folds, and each lead is scored by a copy of the selected model that was trained without that lead. This means the dashboard never shows a score the model produced from memorising a lead it trained on.

Tiers follow FR-04:

| Tier | Probability |
|---|---|
| High | ≥ 0.70 |
| Medium | 0.40 – 0.69 |
| Low | < 0.40 |

In [ ]:
all_probs = cross_val_predict(final_model, X_encoded, y, cv=cv, method='predict_proba', n_jobs=-1)[:, 1]

def assign_tier(prob):
    if prob >= 0.70:
        return 'High'
    elif prob >= 0.40:
        return 'Medium'
    return 'Low'

results_df = df_clean.copy()
results_df['prospect_id'] = lead_ids.values
results_df['conversion_probability'] = all_probs
results_df['priority_tier'] = results_df['conversion_probability'].apply(assign_tier)
results_df = results_df[['prospect_id'] + [c for c in results_df.columns if c != 'prospect_id']]

print("Priority tier distribution:")
print(results_df['priority_tier'].value_counts())
print("\nActual conversion rate in each tier:")
print(results_df.groupby('priority_tier')['Converted'].mean().round(3))
print("\nTotal leads scored:", len(results_df))

## 12. Save the model and outputs
The saved pipeline contains the scaler and the trained model together, so new leads can be scored with a single `predict_proba` call.

In [ ]:
results_df.to_csv('leads_with_predictions.csv', index=False)
df_clean.to_csv('leads_cleaned.csv', index=False)
comparison.to_csv('model_comparison.csv')
joblib.dump(final_model, 'salesmesh_model.pkl')
print("Saved: leads_with_predictions.csv, leads_cleaned.csv, model_comparison.csv, salesmesh_model.pkl")

## 13. Generate SQL to load leads and predictions into MySQL
Colab runs in the cloud and cannot reach a local MySQL server, so the data is written as INSERT statements to a `.sql` file that is run in MySQL Workbench.

In [ ]:
def q(v):
    """Format a value safely for SQL."""
    if pd.isna(v):
        return "NULL"
    return "'" + str(v).replace("\\", "\\\\").replace("'", "''") + "'"

def batches(prefix, rows, size=1000):
    return [prefix + ",\n".join(rows[s:s + size]) + ";" for s in range(0, len(rows), size)]

lines = [
    "USE salesmesh;",
    "SET FOREIGN_KEY_CHECKS = 0;",
    "TRUNCATE TABLE conversion_predictions;",
    "TRUNCATE TABLE leads;",
    "SET FOREIGN_KEY_CHECKS = 1;"
]

lead_rows, pred_rows = [], []
for i, r in results_df.reset_index(drop=True).iterrows():
    lead_id = i + 1
    lead_rows.append(
        f"({lead_id}, {q(r['prospect_id'])}, {q(r['Lead Source'])}, {q(r['Lead Origin'])}, "
        f"{q(r['What is your current occupation'])}, {q(r['City'])}, {int(r['TotalVisits'])}, "
        f"{int(r['Total Time Spent on Website'])}, {q(r['Last Activity'])}, {int(r['Converted'])})"
    )
    pred_rows.append(
        f"({lead_id}, NULL, NULL, {round(float(r['conversion_probability']), 6)}, {q(r['priority_tier'])})"
    )

lines += batches("INSERT INTO leads (lead_id, prospect_id, lead_source, lead_origin, occupation, "
                 "city, total_visits, time_on_website, last_activity, converted) VALUES\n", lead_rows)
lines += batches("INSERT INTO conversion_predictions (lead_id, campaign_id, user_id, "
                 "conversion_probability, priority_tier) VALUES\n", pred_rows)
lines += [
    "SELECT COUNT(*) AS leads_loaded FROM leads;",
    "SELECT priority_tier, COUNT(*) AS total FROM conversion_predictions GROUP BY priority_tier;"
]

with open('load_leads_and_predictions.sql', 'w', encoding='utf-8') as f:
    f.write("\n".join(lines))
print("Created load_leads_and_predictions.sql")

## 14. Load and clean the Marketing Campaign dataset
Upload `marketing_campaign_dataset.csv` when prompted. This data feeds the campaign and ROI views in Power BI, not the ML model, so it needs **no encoding, scaling or SMOTE** — only quality checks and data type fixes:
- `Acquisition_Cost`: `"$16,174.00"` → `16174.0`
- `Duration`: `"30 days"` → `30`

In [ ]:
print("Upload: marketing_campaign_dataset.csv")
uploaded = files.upload()
camp = pd.read_csv(list(uploaded.keys())[0])
print("Shape:", camp.shape)

print("\nMissing values:", camp.isnull().sum().sum())
print("Duplicate rows:", camp.duplicated().sum())
print("Duplicate Campaign_IDs:", camp['Campaign_ID'].duplicated().sum())

camp_clean = camp.drop_duplicates().copy()
camp_clean['Acquisition_Cost'] = (
    camp_clean['Acquisition_Cost'].astype(str)
    .str.replace('$', '', regex=False).str.replace(',', '', regex=False).astype(float)
)
camp_clean['Duration'] = camp_clean['Duration'].astype(str).str.extract(r'(\d+)')[0].astype(int)

camp_clean.to_csv('campaigns_cleaned.csv', index=False)
print("\nRows after cleaning:", len(camp_clean))

## 15. Generate SQL to load campaigns into MySQL
200,000 rows are split into four files of 50,000 so MySQL Workbench does not freeze. Run them in order: part 1 clears the table first.

In [ ]:
rows = [
    f"({int(r.Campaign_ID)}, {q(r.Campaign_Type)}, {q(r.Target_Audience)}, {q(r.Channel_Used)}, "
    f"{int(r.Duration)}, {float(r.Conversion_Rate)}, {float(r.Acquisition_Cost)}, {float(r.ROI)})"
    for r in camp_clean.itertuples(index=False)
]

insert_prefix = ("INSERT INTO campaigns (campaign_id, campaign_type, target_audience, channels_used, "
                 "duration_days, conversion_rate, acquisition_cost, roi) VALUES\n")

parts = [rows[i:i + 50000] for i in range(0, len(rows), 50000)]
campaign_files = []
for n, part in enumerate(parts, start=1):
    part_lines = ["USE salesmesh;"]
    if n == 1:
        part_lines += ["SET FOREIGN_KEY_CHECKS = 0;", "TRUNCATE TABLE campaigns;", "SET FOREIGN_KEY_CHECKS = 1;"]
    part_lines += batches(insert_prefix, part)
    if n == len(parts):
        part_lines += [
            "SELECT COUNT(*) AS campaigns_loaded FROM campaigns;",
            "SELECT campaign_type, ROUND(AVG(roi), 2) AS avg_roi FROM campaigns GROUP BY campaign_type;"
        ]
    fname = f"load_campaigns_part{n}.sql"
    with open(fname, 'w', encoding='utf-8') as f:
        f.write("\n".join(part_lines))
    campaign_files.append(fname)
    print("Created", fname)

## 16. Download all outputs
Save each file to the matching project folder:

| File | Folder |
|---|---|
| leads_with_predictions.csv | predictions/ |
| leads_cleaned.csv, campaigns_cleaned.csv | Data/ |
| salesmesh_model.pkl | models/ |
| model_comparison.csv, model_evaluation.png, feature_importance.png | report/ |
| all `.sql` files | database/ |

In [ ]:
outputs = ['leads_with_predictions.csv', 'leads_cleaned.csv', 'campaigns_cleaned.csv',
           'salesmesh_model.pkl', 'model_comparison.csv', 'model_evaluation.png',
           'feature_importance.png', 'load_leads_and_predictions.sql'] + campaign_files

for f in outputs:
    files.download(f)